**Project Skeleton & Utility Functions**

Our goal in this lesson is not to convert ebooks yet.

Instead, we'll create the foundation that every other lesson will build upon.

# Part I: **Project Skeleton**

## Initial Imports

In [1]:
# ==========================================================
# Universal eBook to PDF Converter v2.0
# ==========================================================

import os
import time
import threading
import subprocess
from pathlib import Path

from google.colab import files

## Application Information

In [2]:
APP_NAME = "Universal eBook to PDF Converter"
VERSION = "2.0"

LINE = "=" * 70

## Global Variables

In [3]:
# Input / Output

input_file = ""
input_path = ""

output_file = ""
output_path = ""


# Conversion Command

command = []


# Progress

progress = 0
conversion_finished = False


# Timing

start_time = None

## Useful Functions

### Header Function

In [4]:
def print_header():

    print("\n" + LINE)
    print(f"{APP_NAME} v{VERSION}".center(70))
    print(LINE)

### Section Function

In [5]:
def print_section(title):

    print("\n")
    print("-" * 70)
    print(title.center(70))
    print("-" * 70)

### Success Message

In [6]:
def success(message):

    print(f"✅ {message}")

In [7]:
# Example
success("Conversion Complete")

✅ Conversion Complete


### Error Message

In [8]:
def error(message):

  print(f"❌ {message}")

### Information Message

In [9]:
def info(message):

    print(f"ℹ️ {message}")

### Warning Message

In [10]:
def warning(message):

    print(f"⚠️ {message}")

### Pause Function

In [11]:
def pause(seconds=1):

    time.sleep(seconds)

### Display Application Banner

In [12]:
print_header()

info("Application Loaded Successfully.")


                Universal eBook to PDF Converter v2.0                 
ℹ️ Application Loaded Successfully.


# Part II: **Configuration Manager**

## Configuration Dictionary

In [13]:
# ==========================================================
# Application Configuration
# ==========================================================

config = {

    # Input / Output
    "input_file": "",
    "input_path": "",
    "output_file": "",
    "output_path": "",

    # PDF Settings
    "paper_size": "a4",

    "margins": {
        "top": 72,
        "bottom": 72,
        "left": 72,
        "right": 72
    },

    "font": {
        "base": 12,
        "pdf_default": 12
    },

    "paragraph": {
        "remove_spacing": False,
        "insert_blank_line": False,
        "blank_line_size": 0.5
    },

    "justify": False,

    # Runtime
    "command": [],

    "progress": 0,

    "conversion_finished": False,

    "start_time": None
}

## Print Configuration

In [14]:
print_section("Current Configuration")

for key, value in config.items():
    print(f"{key:22} : {value}")



----------------------------------------------------------------------
                        Current Configuration                         
----------------------------------------------------------------------
input_file             : 
input_path             : 
output_file            : 
output_path            : 
paper_size             : a4
margins                : {'top': 72, 'bottom': 72, 'left': 72, 'right': 72}
font                   : {'base': 12, 'pdf_default': 12}
paragraph              : {'remove_spacing': False, 'insert_blank_line': False, 'blank_line_size': 0.5}
justify                : False
command                : []
progress               : 0
conversion_finished    : False
start_time             : None


# Part III: **Dependency Manager**

In [15]:
import shutil

In [16]:
def is_calibre_installed():

    return shutil.which("ebook-convert") is not None

In [17]:
def install_calibre():

    print_section("Installing Calibre")

    subprocess.run(
        ["apt-get", "update"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    subprocess.run(
        ["apt-get", "install", "-y", "calibre"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    success("Calibre Installed.")

In [18]:
def setup_dependencies():

    if is_calibre_installed():

        success("Calibre already installed.")

    else:

        install_calibre()

# Part IV: **Interactive User Preference Module**

## Helper Function for Yes/No Questions

In [19]:
def ask_yes_no(question, default=True):
    """
    Ask a Yes/No question and return True or False.
    Pressing Enter selects the default value.
    """

    default_text = "Y/n" if default else "y/N"

    while True:

        choice = input(f"{question} [{default_text}]: ").strip().lower()

        if choice == "":
            return default

        elif choice in ["y", "yes"]:
            return True

        elif choice in ["n", "no"]:
            return False

        else:
            warning("Please enter Y or N.")

## Paper Size Selection

In [20]:
def get_paper_size():

    print_section("Paper Size")

    print("1. A4")
    print("2. Letter")
    print("3. Legal")

    while True:

        choice = input("\nSelect Paper Size [1]: ").strip()

        if choice == "":
            config["paper_size"] = "a4"
            break

        elif choice == "1":
            config["paper_size"] = "a4"
            break

        elif choice == "2":
            config["paper_size"] = "letter"
            break

        elif choice == "3":
            config["paper_size"] = "legal"
            break

        else:
            warning("Invalid choice. Please select 1, 2, or 3.")

    success(f"Paper Size: {config['paper_size'].upper()}")

## Margin Input Helper

In [21]:
def get_margin(name):

    default = config["margins"][name]

    while True:

        value = input(f"{name.capitalize()} Margin [{default}]: ").strip()

        if value == "":
            return default

        try:

            value = int(value)

            if value >= 0:
                return value

            warning("Margin cannot be negative.")

        except ValueError:

            warning("Please enter a valid integer.")

### Margin Settings

In [22]:
def get_margins():

    print_section("Margins")

    config["margins"]["top"] = get_margin("top")
    config["margins"]["bottom"] = get_margin("bottom")
    config["margins"]["left"] = get_margin("left")
    config["margins"]["right"] = get_margin("right")

    success("Margins Updated.")

## Font Settings

In [23]:
def get_font_settings():

    print_section("Font Settings")

    while True:

        value = input(
            f"Base Font Size [{config['font']['base']}]: "
        ).strip()

        if value == "":
            break

        try:

            value = int(value)

            if value > 0:

                config["font"]["base"] = value
                break

            warning("Font size must be positive.")

        except ValueError:

            warning("Please enter a valid integer.")

    success(f"Base Font Size: {config['font']['base']} pt")

## Paragraph Settings

In [24]:
def get_paragraph_settings():

    print_section("Paragraph Settings")

    config["paragraph"]["remove_spacing"] = ask_yes_no(
        "Remove paragraph spacing?",
        default=False
    )

    config["paragraph"]["insert_blank_line"] = ask_yes_no(
        "Insert blank line between paragraphs?",
        default=False
    )

    if config["paragraph"]["insert_blank_line"]:

        while True:

            value = input(
                f"Blank Line Size [{config['paragraph']['blank_line_size']}]: "
            ).strip()

            if value == "":
                break

            try:

                value = float(value)

                if value > 0:

                    config["paragraph"]["blank_line_size"] = value
                    break

                warning("Value must be positive.")

            except ValueError:

                warning("Please enter a valid number.")

    success("Paragraph Settings Updated.")

## Text Justification

In [25]:
def get_justification():

    print_section("Text Alignment")

    config["justify"] = ask_yes_no(
        "Justify text?",
        default=False
    )

    success(
        f"Justification: {'Enabled' if config['justify'] else 'Disabled'}"
    )

## Collect All Preferences

In [26]:
def get_user_preferences():

    print_header()

    info("Configure your PDF output.\n")

    get_paper_size()

    get_margins()

    get_font_settings()

    get_paragraph_settings()

    get_justification()

    print()

    success("All preferences have been collected.")

## Test the Module

In [27]:
get_user_preferences()

print_section("Updated Configuration")

for key, value in config.items():
    print(f"{key:22} : {value}")


                Universal eBook to PDF Converter v2.0                 
ℹ️ Configure your PDF output.



----------------------------------------------------------------------
                              Paper Size                              
----------------------------------------------------------------------
1. A4
2. Letter
3. Legal

Select Paper Size [1]: A4
⚠️ Invalid choice. Please select 1, 2, or 3.

Select Paper Size [1]: 1
✅ Paper Size: A4


----------------------------------------------------------------------
                               Margins                                
----------------------------------------------------------------------
Top Margin [72]: 50
Bottom Margin [72]: 50
Left Margin [72]: 50
Right Margin [72]: 50
✅ Margins Updated.


----------------------------------------------------------------------
                            Font Settings                             
----------------------------------------------------------------------
Base Fo

# Part V: **Upload & Validation Module**

## Supported Formats

In [28]:
# ==========================================================
# Supported eBook Formats
# ==========================================================

SUPPORTED_FORMATS = {
    "epub",
    "mobi",
    "azw",
    "azw3",
    "fb2",
    "html",
    "htm",
    "lit",
    "lrf",
    "odt",
    "pdb",
    "pdf",
    "prc",
    "rb",
    "rtf",
    "snb",
    "tcr",
    "txt"
}

## Upload Function

In [29]:
def upload_file():

    print_section("Upload eBook")

    uploaded = files.upload()

    if not uploaded:

        error("No file uploaded.")

        return False

    filename = next(iter(uploaded))

    config["input_file"] = filename

    config["input_path"] = str(Path(filename).resolve())

    success(f"Uploaded: {filename}")

    return True

## Detect Extension

In [30]:
def detect_extension():

    extension = Path(config["input_file"]).suffix.lower()

    extension = extension.replace(".", "")

    return extension

## Validate File

In [31]:
def validate_file():

    print_section("Validating File")

    extension = detect_extension()

    if extension not in SUPPORTED_FORMATS:

        error(f"Unsupported format: .{extension}")

        print()

        print("Supported formats:")

        print(", ".join(sorted(SUPPORTED_FORMATS)))

        return False

    success(f"Detected Format: {extension.upper()}")

    return True

## Generate Output Filename

In [32]:
def prepare_output_file():

    stem = Path(config["input_file"]).stem

    config["output_file"] = f"{stem}.pdf"

    config["output_path"] = str(
        Path(config["output_file"]).resolve()
    )

    info(f"Output File: {config['output_file']}")

## File Summary

In [33]:
def show_file_information():

    print_section("File Information")

    print(f"Input File : {config['input_file']}")

    print(f"Input Path : {config['input_path']}")

    print(f"Output File: {config['output_file']}")

    print(f"Output Path: {config['output_path']}")

## Upload Workflow

In [34]:
def prepare_file():

    if not upload_file():
        return False

    if not validate_file():
        return False

    prepare_output_file()

    show_file_information()

    return True

## Test the Module

In [36]:
print_header()

if prepare_file():

    success("File is ready for conversion.")


                Universal eBook to PDF Converter v2.0                 


----------------------------------------------------------------------
                             Upload eBook                             
----------------------------------------------------------------------


Saving _OceanofPDF.com_The_Golem_What_you_should_know_about_the_science_-_Harry_Collins_and_Trevor_Pinch.epub to _OceanofPDF.com_The_Golem_What_you_should_know_about_the_science_-_Harry_Collins_and_Trevor_Pinch.epub
✅ Uploaded: _OceanofPDF.com_The_Golem_What_you_should_know_about_the_science_-_Harry_Collins_and_Trevor_Pinch.epub


----------------------------------------------------------------------
                           Validating File                            
----------------------------------------------------------------------
✅ Detected Format: EPUB
ℹ️ Output File: _OceanofPDF.com_The_Golem_What_you_should_know_about_the_science_-_Harry_Collins_and_Trevor_Pinch.pdf


----------------------------------------------------------------------
                           File Information                           
----------------------------------------------------------------------
Input File : _OceanofPDF.com_The_Golem_What_you_should_know_about_the_science_-_Harry_Collins_and

# Part VI: **Command Builder Module**

## Base Command

In [37]:
def build_base_command():

    config["command"] = [

        "ebook-convert",

        config["input_path"],

        config["output_path"]

    ]

## Custom Commands

### Paper Size

In [38]:
def add_paper_size():

    config["command"].extend([

        "--paper-size",

        config["paper_size"]

    ])

### Margins

In [39]:
def add_margins():

    margins = config["margins"]

    config["command"].extend([

        "--margin-top",
        str(margins["top"]),

        "--margin-bottom",
        str(margins["bottom"]),

        "--margin-left",
        str(margins["left"]),

        "--margin-right",
        str(margins["right"])

    ])

### Font Settings

In [40]:
def add_font_settings():

    font = config["font"]

    config["command"].extend([

        "--base-font-size",

        str(font["base"])

    ])

### Paragraph Settings

In [41]:
def add_paragraph_settings():

    paragraph = config["paragraph"]

    if paragraph["remove_spacing"]:

        config["command"].append(
            "--remove-paragraph-spacing"
        )

    if paragraph["insert_blank_line"]:

        config["command"].extend([

            "--insert-blank-line",

            "--insert-blank-line-size",

            str(paragraph["blank_line_size"])

        ])

### Justification

In [42]:
def create_custom_css():

    if not config["justify"]:

        return None

    css = """
body{
    text-align:justify;
}
"""

    css_file = "custom_style.css"

    with open(css_file, "w") as file:

        file.write(css)

    return css_file

In [43]:
def add_custom_css():

    css_file = create_custom_css()

    if css_file is None:

        return

    config["command"].extend([

        "--extra-css",

        css_file

    ])

## Display the Command

In [44]:
def show_command():

    print_section("Generated Command")

    print()

    print(" ".join(config["command"]))

## Complete Command

In [45]:
def build_command():

    print_section("Building Command")

    build_base_command()

    add_paper_size()

    add_margins()

    add_font_settings()

    add_paragraph_settings()

    add_custom_css()

    success("Command Generated.")

## Test Everything

In [46]:
build_command()

show_command()



----------------------------------------------------------------------
                           Building Command                           
----------------------------------------------------------------------
✅ Command Generated.


----------------------------------------------------------------------
                          Generated Command                           
----------------------------------------------------------------------

ebook-convert /content/_OceanofPDF.com_The_Golem_What_you_should_know_about_the_science_-_Harry_Collins_and_Trevor_Pinch.epub /content/_OceanofPDF.com_The_Golem_What_you_should_know_about_the_science_-_Harry_Collins_and_Trevor_Pinch.pdf --paper-size a4 --margin-top 50 --margin-bottom 50 --margin-left 50 --margin-right 50 --base-font-size 12 --insert-blank-line --insert-blank-line-size 0.5 --extra-css custom_style.css


# Part VII: **Conversion Engine**

## Prepare the Environment

In [47]:
def prepare_environment():

    env = os.environ.copy()

    env["QTWEBENGINE_DISABLE_SANDBOX"] = "1"

    return env

## Start the Conversion Timer

In [48]:
def start_timer():

    config["start_time"] = time.time()

## Stop the Timer

In [49]:
def stop_timer():

    elapsed = time.time() - config["start_time"]

    return round(elapsed, 2)

## Execute Conversion

In [50]:
def execute_conversion():

    env = prepare_environment()

    start_timer()

    result = subprocess.run(

        config["command"],

        capture_output=True,

        text=True,

        env=env

    )

    elapsed = stop_timer()

    return result, elapsed

## Handle the Success / Failure

In [51]:
def conversion_success(elapsed):

    success("Conversion completed successfully.")

    print()

    print(f"Time Taken : {elapsed:.2f} seconds")

In [52]:
def conversion_failure(result):

    error("Conversion Failed")

    print()

    print(result.stderr)

## Main Conversion Function

In [53]:
def convert():

    print_section("Converting eBook")

    if not config["command"]:

        error("Conversion command has not been built.")

        return False

    if not os.path.exists(config["input_path"]):

        error("Input file not found.")

        return False

    result, elapsed = execute_conversion()

    if result.returncode == 0:

        conversion_success(elapsed)

        return True

    conversion_failure(result)

    return False

## Test

In [54]:
setup_dependencies()
convert()



----------------------------------------------------------------------
                          Installing Calibre                          
----------------------------------------------------------------------
✅ Calibre Installed.


----------------------------------------------------------------------
                           Converting eBook                           
----------------------------------------------------------------------
✅ Conversion completed successfully.

Time Taken : 23.10 seconds


True

# Part VIII: **Progress Manager**

## Initilize Progress State

In [55]:
import threading

In [56]:
def reset_progress():

    config["progress"] = 0

    config["conversion_finished"] = False

## Define Conversion Stages

In [57]:
PROGRESS_STAGES = [

    "Initializing Converter...",

    "Reading eBook...",

    "Processing Chapters...",

    "Formatting Pages...",

    "Embedding Fonts...",

    "Rendering PDF...",

    "Optimizing Output...",

    "Finalizing PDF..."

]

## Progress Bar Function

In [58]:
def progress_bar():

    stage = 0

    total_stages = len(PROGRESS_STAGES)

    while not config["conversion_finished"]:

        if config["progress"] < 95:

            config["progress"] += 1

        current_stage = min(

            stage,

            total_stages - 1

        )

        elapsed = 0

        if config["start_time"] is not None:

            elapsed = int(time.time() - config["start_time"])

        filled = config["progress"] // 5

        bar = "█" * filled + "░" * (20 - filled)

        print(

            f"\r[{bar}] "

            f"{config['progress']:3d}%   "

            f"{PROGRESS_STAGES[current_stage]}   "

            f"Elapsed: {elapsed}s",

            end=""

        )

        if config["progress"] % 12 == 0:

            stage += 1

        time.sleep(0.20)

    config["progress"] = 100

    bar = "█" * 20

    elapsed = int(time.time() - config["start_time"])

    print(

        f"\r[{bar}] 100%   Conversion Complete!   Elapsed: {elapsed}s"
    )

## Start Progress Manager

In [59]:
def start_progress():

    reset_progress()

    thread = threading.Thread(

        target=progress_bar,

        daemon=True

    )

    thread.start()

    return thread

## Stop Progress Manager

In [60]:
def stop_progress(thread):

    config["conversion_finished"] = True

    thread.join()

## Progress Demonstration

In [61]:
config["start_time"] = time.time()

thread = start_progress()

time.sleep(8)

stop_progress(thread)

[████████████████████] 100%   Conversion Complete!   Elapsed: 8s


## Integrate with the Conversion Engine

In [62]:
def execute_conversion():

    env = prepare_environment()

    start_timer()

    progress_thread = start_progress()

    result = subprocess.run(

        config["command"],

        capture_output=True,

        text=True,

        env=env

    )

    stop_progress(progress_thread)

    elapsed = stop_timer()

    return result, elapsed

# Part IX: **Verification and Conversion Report**

## Verify Output

In [63]:
def verify_output():

    print_section("Verifying Output")

    if os.path.exists(config["output_path"]):

        success("Output PDF verified.")

        return True

    error("Output PDF was not created.")

    return False

## File Size

In [64]:
def get_file_size():

    size = os.path.getsize(config["output_path"])

    size_mb = size / (1024 * 1024)

    return round(size_mb, 2)

## Display Conversion Report

In [65]:
def generate_report():

    print_section("Conversion Report")

    print(f"Input File        : {config['input_file']}")

    print(f"Output File       : {config['output_file']}")

    print(f"Paper Size        : {config['paper_size'].upper()}")

    print()

    print("Margins")

    print("-------")

    print(f"Top               : {config['margins']['top']}")

    print(f"Bottom            : {config['margins']['bottom']}")

    print(f"Left              : {config['margins']['left']}")

    print(f"Right             : {config['margins']['right']}")

    print()

    print("Font")

    print("----")

    print(f"Base Font         : {config['font']['base']} pt")

    print()

    print("Paragraph")

    print("---------")

    print(f"Remove Spacing    : {config['paragraph']['remove_spacing']}")

    print(f"Blank Line        : {config['paragraph']['insert_blank_line']}")

    print(f"Justification     : {config['justify']}")

    print()

    print(f"Output Size       : {get_file_size()} MB")

## Download PDF

In [66]:
def download_pdf():

    print_section("Download")

    files.download(config["output_path"])

    success("Download Started.")

## Finalize Conversion

In [67]:
def finalize_conversion():

    if not verify_output():

        return

    generate_report()

    download_pdf()

## Test

In [68]:
finalize_conversion()



----------------------------------------------------------------------
                           Verifying Output                           
----------------------------------------------------------------------
✅ Output PDF verified.


----------------------------------------------------------------------
                          Conversion Report                           
----------------------------------------------------------------------
Input File        : _OceanofPDF.com_The_Golem_What_you_should_know_about_the_science_-_Harry_Collins_and_Trevor_Pinch.epub
Output File       : _OceanofPDF.com_The_Golem_What_you_should_know_about_the_science_-_Harry_Collins_and_Trevor_Pinch.pdf
Paper Size        : A4

Margins
-------
Top               : 50
Bottom            : 50
Left              : 50
Right             : 50

Font
----
Base Font         : 12 pt

Paragraph
---------
Remove Spacing    : False
Blank Line        : True
Justification     : True

Output Size       : 1.47 MB


-----

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download Started.


# Part X: **Main Controller**

## Main Function

In [69]:
def main():

    print_header()

    # --------------------------------------------------
    # Step 1 : Check Dependencies
    # --------------------------------------------------

    setup_dependencies()

    print()

    # --------------------------------------------------
    # Step 2 : Upload & Validate File
    # --------------------------------------------------

    if not prepare_file():

        error("Application terminated.")

        return

    print()

    # --------------------------------------------------
    # Step 3 : Collect User Preferences
    # --------------------------------------------------

    get_user_preferences()

    print()

    # --------------------------------------------------
    # Step 4 : Build Conversion Command
    # --------------------------------------------------

    build_command()

    print()

    show_command()

    print()

    # --------------------------------------------------
    # Step 5 : Convert
    # --------------------------------------------------

    if not convert():

        error("Conversion unsuccessful.")

        return

    print()

    # --------------------------------------------------
    # Step 6 : Generate Report
    # --------------------------------------------------

    finalize_conversion()

    print()

    success("Application Finished Successfully.")

## Launch the Application

In [71]:
main()


                Universal eBook to PDF Converter v2.0                 
✅ Calibre already installed.



----------------------------------------------------------------------
                             Upload eBook                             
----------------------------------------------------------------------


Saving _OceanofPDF.com_The_Golem_What_you_should_know_about_the_science_-_Harry_Collins_and_Trevor_Pinch.epub to _OceanofPDF.com_The_Golem_What_you_should_know_about_the_science_-_Harry_Collins_and_Trevor_Pinch (2).epub
✅ Uploaded: _OceanofPDF.com_The_Golem_What_you_should_know_about_the_science_-_Harry_Collins_and_Trevor_Pinch (2).epub


----------------------------------------------------------------------
                           Validating File                            
----------------------------------------------------------------------
✅ Detected Format: EPUB
ℹ️ Output File: _OceanofPDF.com_The_Golem_What_you_should_know_about_the_science_-_Harry_Collins_and_Trevor_Pinch (2).pdf


----------------------------------------------------------------------
                           File Information                           
----------------------------------------------------------------------
Input File : _OceanofPDF.com_The_Golem_What_you_should_know_about_the_science_-_Harry

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download Started.

✅ Application Finished Successfully.


In [72]:
!pip install -q gradio
setup_dependencies()

✅ Calibre already installed.


In [73]:
import os
import time
import html
import inspect
import subprocess
from pathlib import Path
import gradio as gr

In [74]:
OUTPUT_DIR = Path("/content/converted_pdfs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [75]:
# Paper dimensions in points (72 pt = 1 inch)
PAPER_DIMENSIONS_PT = {
    "a4": (595, 842),
    "letter": (612, 792),
    "legal": (612, 1008),
}

PRESETS = {
    "fiction": {
        "label": "📖 Fiction & Novels",
        "paper_size": "A4",
        "margins": (54, 54, 54, 54),
        "font": 13,
        "remove_spacing": False,
        "insert_blank": True,
        "blank_size": 0.5,
        "justify": True,
        "desc": "Balanced 0.75\" margins, justified alignment, and comfortable 13 pt reading flow."
    },
    "academic": {
        "label": "🎓 Technical & Academic",
        "paper_size": "A4",
        "margins": (36, 36, 36, 36),
        "font": 11,
        "remove_spacing": False,
        "insert_blank": True,
        "blank_size": 0.4,
        "justify": False,
        "desc": "Compact 0.5\" margins and left-aligned 11 pt text to preserve code blocks and equations."
    },
    "print": {
        "label": "🖨️ Classic Print (1\" Margins)",
        "paper_size": "Letter",
        "margins": (72, 72, 72, 72),
        "font": 12,
        "remove_spacing": False,
        "insert_blank": False,
        "blank_size": 0.5,
        "justify": True,
        "desc": "Standard US Letter with full 1-inch margins and classic book justification."
    },
    "comfort": {
        "label": "📱 Large Print / Comfort",
        "paper_size": "A4",
        "margins": (32, 32, 32, 32),
        "font": 16,
        "remove_spacing": False,
        "insert_blank": True,
        "blank_size": 0.8,
        "justify": True,
        "desc": "Generous 16 pt typography with extra paragraph breathing room for effortless reading."
    }
}

In [76]:
def _extract_filepath(file_obj):
    if file_obj is None:
        return None
    return file_obj.name if hasattr(file_obj, "name") else str(file_obj)

In [77]:
def render_header_banner():
    calibre_ok = is_calibre_installed()
    status_bg = "rgba(16, 185, 129, 0.16)" if calibre_ok else "rgba(245, 158, 11, 0.18)"
    status_border = "#10b981" if calibre_ok else "#f59e0b"
    status_text = "🟢 Calibre Engine Ready" if calibre_ok else "🟡 Calibre Not Detected (Run setup_dependencies)"

    return f"""
    <div style="
        background: linear-gradient(135deg, #0f172a 0%, #1e293b 55%, #312e81 100%);
        border-radius: 16px;
        padding: 24px 28px;
        color: #f8fafc;
        box-shadow: 0 10px 25px -5px rgba(15, 23, 42, 0.25);
        margin-bottom: 8px;
        border: 1px solid rgba(255,255,255,0.08);
    ">
        <div style="display: flex; flex-wrap: wrap; justify-content: space-between; align-items: center; gap: 12px;">
            <div>
                <div style="font-size: 12px; letter-spacing: 1.4px; text-transform: uppercase; color: #a5b4fc; font-weight: 700;">
                    Publishing & Typography Studio
                </div>
                <h1 style="margin: 4px 0 6px 0; font-size: 26px; font-weight: 800; color: #ffffff;">
                    📚 {html.escape(APP_NAME)} <span style="font-size: 14px; background: #4f46e5; padding: 3px 10px; border-radius: 999px; vertical-align: middle;">v{html.escape(VERSION)}</span>
                </h1>
                <p style="margin: 0; color: #cbd5e1; font-size: 14px;">
                    Convert <b>{len(SUPPORTED_FORMATS)} eBook formats</b> into print-grade PDFs with live layout simulation and smart presets.
                </p>
            </div>
            <div style="display: flex; flex-direction: column; align-items: flex-end; gap: 8px;">
                <span style="
                    background: {status_bg};
                    border: 1px solid {status_border};
                    color: #ffffff;
                    padding: 6px 14px;
                    border-radius: 999px;
                    font-size: 13px;
                    font-weight: 600;
                ">{status_text}</span>
                <span style="font-size: 12px; color: #94a3b8;">
                    EPUB · MOBI · AZW3 · FB2 · ODT · RTF · TXT & more
                </span>
            </div>
        </div>
    </div>
    """

In [78]:
def inspect_uploaded_file(uploaded_file):
    path_str = _extract_filepath(uploaded_file)
    if not path_str or not os.path.exists(path_str):
        return """
        <div style="padding: 12px 16px; border-radius: 10px; border: 1px dashed #94a3b8; color: #64748b; font-size: 13px; text-align: center;">
            ⬆️ Drop any eBook above to inspect format compatibility and estimated conversion time.
        </div>
        """

    p = Path(path_str)
    ext = p.suffix.lower().lstrip(".")
    size_bytes = os.path.getsize(p)
    size_str = f"{size_bytes / (1024 * 1024):.2f} MB" if size_bytes >= 1024 * 1024 else f"{size_bytes / 1024:.1f} KB"

    if ext not in SUPPORTED_FORMATS:
        return f"""
        <div style="padding: 14px 16px; border-radius: 10px; background: rgba(239, 68, 68, 0.1); border: 1px solid #ef4444; color: #b91c1c; font-size: 13px;">
            <b>❌ Unsupported Format (.{html.escape(ext)})</b><br>
            Supported formats: <code>{', '.join(sorted(SUPPORTED_FORMATS))}</code>
        </div>
        """

    est_sec = max(8, int((size_bytes / (1024 * 1024)) * 12) + 8)
    safe_name = html.escape(p.name)
    safe_out = html.escape(f"{p.stem}.pdf")

    return f"""
    <div style="padding: 14px 16px; border-radius: 12px; background: rgba(16, 185, 129, 0.08); border: 1px solid rgba(16, 185, 129, 0.45); font-size: 13px;">
        <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 6px;">
            <span style="font-weight: 700; color: #059669;">✅ Ready for Conversion</span>
            <span style="background: #059669; color: white; padding: 2px 9px; border-radius: 6px; font-weight: 700; font-size: 11px;">{ext.upper()}</span>
        </div>
        <div style="line-height: 1.6;">
            <div><b>Source:</b> <code>{safe_name}</code> ({size_str})</div>
            <div><b>Target Output:</b> <code>{safe_out}</code> · <b>Est. Time:</b> ~{est_sec}s</div>
        </div>
    </div>
    """

In [79]:
def build_live_previews(
    uploaded_file,
    paper_size,
    m_top,
    m_bottom,
    m_left,
    m_right,
    base_font,
    remove_spacing,
    insert_blank,
    blank_line_size,
    justify
):
    paper_key = (paper_size or "A4").lower()
    pw, ph = PAPER_DIMENSIONS_PT.get(paper_key, (595, 842))

    # Scale paper sheet visually to ~260px width while preserving exact aspect ratio
    preview_w = 260
    scale = preview_w / pw
    preview_h = int(ph * scale)

    pad_t = max(2, int((m_top or 0) * scale))
    pad_b = max(2, int((m_bottom or 0) * scale))
    pad_l = max(2, int((m_left or 0) * scale))
    pad_r = max(2, int((m_right or 0) * scale))

    font_px = max(6.5, round((base_font or 12) * scale * 1.45, 1))
    align_css = "justify" if justify else "left"

    if remove_spacing:
        p_margin_bottom = "0px"
    elif insert_blank:
        p_margin_bottom = f"{round((blank_line_size or 0.5) * font_px, 1)}px"
    else:
        p_margin_bottom = f"{round(font_px * 0.45, 1)}px"

    page_html = f"""
    <div style="
        background: linear-gradient(180deg, #1e293b 0%, #0f172a 100%);
        border-radius: 14px;
        padding: 18px 14px;
        display: flex;
        flex-direction: column;
        align-items: center;
        border: 1px solid rgba(148, 163, 184, 0.25);
    ">
        <div style="color: #cbd5e1; font-size: 12px; font-weight: 600; margin-bottom: 12px; letter-spacing: 0.5px;">
            👁️ LIVE PAGE & TYPOGRAPHY SIMULATOR ({paper_key.upper()})
        </div>

        <!-- Simulated Paper Sheet -->
        <div style="
            width: {preview_w}px;
            height: {preview_h}px;
            background: #fffdf9;
            color: #1e293b;
            box-shadow: 0 14px 30px rgba(0,0,0,0.45);
            border-radius: 3px;
            box-sizing: border-box;
            padding: {pad_t}px {pad_r}px {pad_b}px {pad_l}px;
            overflow: hidden;
            position: relative;
        ">
            <!-- Margin Guide Box -->
            <div style="
                width: 100%;
                height: 100%;
                border: 1px dashed rgba(99, 102, 241, 0.45);
                box-sizing: border-box;
                padding: 4px;
                overflow: hidden;
                font-family: Georgia, 'Times New Roman', serif;
                font-size: {font_px}px;
                line-height: 1.42;
                text-align: {align_css};
            ">
                <div style="font-weight: 700; font-size: {round(font_px * 1.25, 1)}px; margin-bottom: {round(font_px * 0.6, 1)}px; text-align: left; color: #0f172a;">
                    Chapter I · The First Page
                </div>
                <p style="margin: 0 0 {p_margin_bottom} 0;">
                    Typography shapes how effortlessly a reader absorbs complex ideas. Adjusting margins and base font size gives every page clear visual rhythm and comfortable line length.
                </p>
                <p style="margin: 0 0 {p_margin_bottom} 0;">
                    Notice how paragraph spacing and text alignment respond immediately as you refine your layout settings on the left.
                </p>
                <p style="margin: 0;">
                    Whether you are preparing a dense technical reference or a relaxing evening novel, your PDF will render cleanly at {int(base_font)} pt on {paper_key.upper()} paper.
                </p>
            </div>
        </div>

        <!-- Live Spec Pill -->
        <div style="
            margin-top: 12px;
            background: rgba(255,255,255,0.08);
            color: #e2e8f0;
            padding: 6px 12px;
            border-radius: 999px;
            font-size: 11.5px;
            text-align: center;
        ">
            {paper_key.upper()} ({pw}×{ph} pt) · Font: <b>{int(base_font)} pt</b> · Margins: <b>{int(m_top)}/{int(m_bottom)}/{int(m_left)}/{int(m_right)}</b> · Align: <b>{'Justified' if justify else 'Left'}</b>
        </div>
    </div>
    """

    # Build a live preview of the Calibre CLI flags
    path_str = _extract_filepath(uploaded_file)
    in_name = Path(path_str).name if path_str else "<uploaded_ebook.epub>"
    out_name = f"{Path(in_name).stem}.pdf" if path_str else "<output.pdf>"

    flags = [
        f'ebook-convert "{in_name}" "{out_name}"',
        f"--paper-size {paper_key}",
        f"--margin-top {int(m_top)} --margin-bottom {int(m_bottom)} --margin-left {int(m_left)} --margin-right {int(m_right)}",
        f"--base-font-size {int(base_font)}",
    ]
    if remove_spacing:
        flags.append("--remove-paragraph-spacing")
    if insert_blank:
        flags.append(f"--insert-blank-line --insert-blank-line-size {float(blank_line_size)}")
    if justify:
        flags.append("--extra-css custom_style.css")

    cli_md = "```bash\n" + " \\\n  ".join(flags) + "\n```"
    return page_html, cli_md

In [80]:
def apply_preset(preset_key):
    p = PRESETS[preset_key]
    gr.Info(f"Applied preset: {p['label']} — {p['desc']}")
    mt, mb, ml, mr = p["margins"]
    return (
        p["paper_size"],
        True,  # sync_margins
        mt,
        gr.update(value=mb, interactive=False),
        gr.update(value=ml, interactive=False),
        gr.update(value=mr, interactive=False),
        p["font"],
        p["remove_spacing"],
        p["insert_blank"],
        gr.update(value=p["blank_size"], interactive=p["insert_blank"]),
        p["justify"],
    )

In [81]:
def reset_to_defaults():
    gr.Info("Reset all formatting options to default values.")
    return (
        "A4",
        True,
        72,
        gr.update(value=72, interactive=False),
        gr.update(value=72, interactive=False),
        gr.update(value=72, interactive=False),
        12,
        False,
        False,
        gr.update(value=0.5, interactive=False),
        False,
    )

In [82]:
def on_toggle_margin_sync(sync_enabled, top_val):
    if sync_enabled:
        return (
            gr.update(value=top_val, interactive=False),
            gr.update(value=top_val, interactive=False),
            gr.update(value=top_val, interactive=False),
        )
    return (
        gr.update(interactive=True),
        gr.update(interactive=True),
        gr.update(interactive=True),
    )

In [83]:
def on_top_margin_change(sync_enabled, top_val):
    if sync_enabled:
        return top_val, top_val, top_val
    return gr.update(), gr.update(), gr.update()

In [84]:
def set_quick_margins(val):
    return (
        True,
        val,
        gr.update(value=val, interactive=False),
        gr.update(value=val, interactive=False),
        gr.update(value=val, interactive=False),
    )

In [85]:
def run_conversion_ui(
    uploaded_file,
    paper_size,
    margin_top,
    margin_bottom,
    margin_left,
    margin_right,
    base_font_size,
    remove_spacing,
    insert_blank_line,
    blank_line_size,
    justify_text,
    history_records,
    progress=gr.Progress()
):
    path_str = _extract_filepath(uploaded_file)
    if not path_str or not os.path.exists(path_str):
        raise gr.Error("Please upload an eBook file before starting conversion.")

    input_path = Path(path_str)
    ext = input_path.suffix.lower().lstrip(".")

    if ext not in SUPPORTED_FORMATS:
        raise gr.Error(
            f"Unsupported format '.{ext}'. Supported formats: {', '.join(sorted(SUPPORTED_FORMATS))}"
        )

    if not is_calibre_installed():
        progress(0.05, desc="Installing Calibre engine (first-time setup)...")
        setup_dependencies()

    # Populate existing global config dictionary cleanly
    output_filename = f"{input_path.stem}.pdf"
    output_path = OUTPUT_DIR / output_filename

    config["input_file"] = input_path.name
    config["input_path"] = str(input_path.resolve())
    config["output_file"] = output_filename
    config["output_path"] = str(output_path.resolve())

    config["paper_size"] = paper_size.lower()
    config["margins"]["top"] = int(margin_top)
    config["margins"]["bottom"] = int(margin_bottom)
    config["margins"]["left"] = int(margin_left)
    config["margins"]["right"] = int(margin_right)
    config["font"]["base"] = int(base_font_size)

    config["paragraph"]["remove_spacing"] = bool(remove_spacing)
    config["paragraph"]["insert_blank_line"] = bool(insert_blank_line)
    config["paragraph"]["blank_line_size"] = float(blank_line_size)
    config["justify"] = bool(justify_text)

    # Use existing build_command() and prepare_environment()
    progress(0.12, desc="Building Calibre conversion command...")
    build_command()
    env = prepare_environment()

    start_t = time.time()
    process = subprocess.Popen(
        config["command"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        env=env
    )

    stage_idx = 0
    sim_progress = 0.16
    while process.poll() is None:
        elapsed = int(time.time() - start_t)
        stage_label = PROGRESS_STAGES[min(stage_idx, len(PROGRESS_STAGES) - 1)]
        progress(sim_progress, desc=f"{stage_label} ({elapsed}s elapsed)")
        if sim_progress < 0.93:
            sim_progress += 0.025
        if elapsed % 3 == 0:
            stage_idx += 1
        time.sleep(0.5)

    _, stderr = process.communicate()
    total_time = round(time.time() - start_t, 2)

    if process.returncode != 0 or not output_path.exists():
        raise gr.Error(f"Conversion failed:\n{stderr}")

    progress(1.0, desc="PDF ready!")
    gr.Info(f"✅ Converted '{output_filename}' in {total_time}s!")

    in_size_mb = round(os.path.getsize(input_path) / (1024 * 1024), 2)
    out_size_mb = round(os.path.getsize(output_path) / (1024 * 1024), 2)
    abs_out = str(output_path.resolve())

    # Rich HTML Summary Dashboard
    report_html = f"""
    <div style="border-radius: 14px; border: 1px solid rgba(16, 185, 129, 0.4); background: rgba(16, 185, 129, 0.06); padding: 18px;">
        <div style="display: flex; align-items: center; gap: 10px; margin-bottom: 14px;">
            <span style="font-size: 22px;">🎉</span>
            <div>
                <div style="font-weight: 800; font-size: 16px; color: #059669;">Conversion Complete & Verified</div>
                <div style="font-size: 12.5px; opacity: 0.85;">{html.escape(output_filename)}</div>
            </div>
        </div>

        <div style="display: grid; grid-template-columns: repeat(3, 1fr); gap: 10px; margin-bottom: 14px;">
            <div style="background: rgba(255,255,255,0.65); color: #0f172a; padding: 10px 12px; border-radius: 10px; border: 1px solid rgba(148,163,184,0.3);">
                <div style="font-size: 11px; text-transform: uppercase; color: #64748b; font-weight: 700;">PDF Size</div>
                <div style="font-size: 18px; font-weight: 800;">{out_size_mb} MB</div>
                <div style="font-size: 11px; color: #64748b;">from {in_size_mb} MB {ext.upper()}</div>
            </div>
            <div style="background: rgba(255,255,255,0.65); color: #0f172a; padding: 10px 12px; border-radius: 10px; border: 1px solid rgba(148,163,184,0.3);">
                <div style="font-size: 11px; text-transform: uppercase; color: #64748b; font-weight: 700;">Render Time</div>
                <div style="font-size: 18px; font-weight: 800;">{total_time}s</div>
                <div style="font-size: 11px; color: #64748b;">Calibre QtWebEngine</div>
            </div>
            <div style="background: rgba(255,255,255,0.65); color: #0f172a; padding: 10px 12px; border-radius: 10px; border: 1px solid rgba(148,163,184,0.3);">
                <div style="font-size: 11px; text-transform: uppercase; color: #64748b; font-weight: 700;">Typography</div>
                <div style="font-size: 18px; font-weight: 800;">{int(base_font_size)} pt · {paper_size.upper()}</div>
                <div style="font-size: 11px; color: #64748b;">{'Justified' if justify_text else 'Left-aligned'}</div>
            </div>
        </div>

        <table style="width: 100%; border-collapse: collapse; font-size: 13px;">
            <tr style="border-bottom: 1px solid rgba(148,163,184,0.25);">
                <td style="padding: 6px 4px; font-weight: 600;">Margins (T / B / L / R)</td>
                <td style="padding: 6px 4px; text-align: right;">{int(margin_top)} / {int(margin_bottom)} / {int(margin_left)} / {int(margin_right)} pt</td>
            </tr>
            <tr style="border-bottom: 1px solid rgba(148,163,184,0.25);">
                <td style="padding: 6px 4px; font-weight: 600;">Remove Paragraph Spacing</td>
                <td style="padding: 6px 4px; text-align: right;">{'Yes' if remove_spacing else 'No'}</td>
            </tr>
            <tr>
                <td style="padding: 6px 4px; font-weight: 600;">Insert Blank Line</td>
                <td style="padding: 6px 4px; text-align: right;">{f'Yes ({blank_line_size} em)' if insert_blank_line else 'No'}</td>
            </tr>
        </table>
    </div>
    """

    # Inline PDF Reader HTML
    major_ver = int(gr.__version__.split(".")[0]) if gr.__version__[0].isdigit() else 5
    file_route = "/gradio_api/file=" if major_ver >= 5 else "/file="
    pdf_url = f"{file_route}{abs_out}"

    pdf_viewer_html = f"""
    <div style="border-radius: 12px; overflow: hidden; border: 1px solid rgba(148,163,184,0.35);">
        <div style="background: #1e293b; color: #f8fafc; padding: 8px 14px; font-size: 12.5px; display: flex; justify-content: space-between; align-items: center;">
            <span>📖 <b>{html.escape(output_filename)}</b> ({out_size_mb} MB)</span>
            <a href="{pdf_url}" target="_blank" style="color: #93c5fd; text-decoration: none; font-weight: 600;">Open in Full Tab ↗</a>
        </div>
        <iframe src="{pdf_url}" width="100%" height="560px" style="border: none; background: #334155;"></iframe>
    </div>
    """

    # Update Session History
    history_records = list(history_records or [])
    timestamp = time.strftime("%H:%M:%S")
    history_records.insert(0, [
        timestamp,
        input_path.name,
        output_filename,
        f"{paper_size.upper()} / {int(base_font_size)}pt",
        f"{out_size_mb} MB",
        f"{total_time}s"
    ])

    return abs_out, report_html, pdf_viewer_html, history_records, history_records

In [86]:
# ==========================================================
# Build Gradio Studio Layout
# ==========================================================
CUSTOM_CSS = """
.preset-btn {
    border-radius: 10px !important;
    font-weight: 600 !important;
    transition: transform 0.15s ease, box-shadow 0.15s ease !important;
}
.preset-btn:hover {
    transform: translateY(-1px);
    box-shadow: 0 4px 12px rgba(79, 70, 229, 0.2) !important;
}
.convert-cta {
    background: linear-gradient(135deg, #4f46e5 0%, #6366f1 100%) !important;
    border: none !important;
    font-size: 16px !important;
    font-weight: 700 !important;
    letter-spacing: 0.3px !important;
    border-radius: 12px !important;
}
"""

In [87]:
file_types = [f".{ext}" for ext in sorted(SUPPORTED_FORMATS)]
initial_preview_html, initial_cli_md = build_live_previews(
    None, "A4", 72, 72, 72, 72, 12, False, False, 0.5, False
)

In [88]:
blocks_kwargs = {"title": f"{APP_NAME} v{VERSION}"}
if "theme" in inspect.signature(gr.Blocks.__init__).parameters:
    blocks_kwargs["theme"] = gr.themes.Soft(primary_hue="indigo", neutral_hue="slate")
    blocks_kwargs["css"] = CUSTOM_CSS

In [89]:
with gr.Blocks(**blocks_kwargs) as demo:
    history_state = gr.State([])

    # Top Hero Header
    gr.HTML(render_header_banner())

    with gr.Row(equal_height=False):
        # ==================================================
        # LEFT COLUMN: Upload, Smart Presets & Customization
        # ==================================================
        with gr.Column(scale=6):
            with gr.Group():
                gr.Markdown("### 1. Upload Source eBook")
                file_input = gr.File(
                    label="Drag & drop your eBook here (EPUB, MOBI, AZW3, FB2, ODT, TXT, etc.)",
                    file_types=file_types,
                    type="filepath"
                )
                file_status_html = gr.HTML(inspect_uploaded_file(None))

            with gr.Group():
                gr.Markdown("### 2. Quick Reading Presets *(One-Click Setup)*")
                with gr.Row():
                    btn_fiction = gr.Button(PRESETS["fiction"]["label"], size="sm", elem_classes=["preset-btn"])
                    btn_academic = gr.Button(PRESETS["academic"]["label"], size="sm", elem_classes=["preset-btn"])
                with gr.Row():
                    btn_print = gr.Button(PRESETS["print"]["label"], size="sm", elem_classes=["preset-btn"])
                    btn_comfort = gr.Button(PRESETS["comfort"]["label"], size="sm", elem_classes=["preset-btn"])

            with gr.Group():
                with gr.Row():
                    gr.Markdown("### 3. Fine-Tune Page & Typography")
                    btn_reset = gr.Button("↺ Reset Defaults", size="sm", variant="secondary")

                with gr.Row():
                    paper_size = gr.Radio(
                        choices=["A4", "Letter", "Legal"],
                        value="A4",
                        label="Paper Size"
                    )
                    base_font = gr.Slider(
                        minimum=8,
                        maximum=28,
                        value=12,
                        step=1,
                        label="Base Font Size (pt)"
                    )

                with gr.Row():
                    sync_margins = gr.Checkbox(value=True, label="🔗 Sync all 4 margins")
                    btn_m_compact = gr.Button("Compact (36 pt)", size="sm")
                    btn_m_standard = gr.Button("Standard (54 pt)", size="sm")
                    btn_m_wide = gr.Button("Wide / 1\" (72 pt)", size="sm")

                with gr.Row():
                    m_top = gr.Number(value=72, label="Top / All (pt)", precision=0, minimum=0)
                    m_bottom = gr.Number(value=72, label="Bottom (pt)", precision=0, minimum=0, interactive=False)
                    m_left = gr.Number(value=72, label="Left (pt)", precision=0, minimum=0, interactive=False)
                    m_right = gr.Number(value=72, label="Right (pt)", precision=0, minimum=0, interactive=False)

            with gr.Group():
                gr.Markdown("### 4. Paragraph Flow & Alignment")
                with gr.Row():
                    justify = gr.Checkbox(value=False, label="Justify Text")
                    remove_spacing = gr.Checkbox(value=False, label="Remove Paragraph Spacing")
                    insert_blank = gr.Checkbox(value=False, label="Insert Blank Line")

                blank_line_size = gr.Slider(
                    minimum=0.2,
                    maximum=3.0,
                    value=0.5,
                    step=0.1,
                    label="Blank Line Size (em)",
                    interactive=False
                )

            convert_btn = gr.Button(
                "🚀 Convert eBook to PDF",
                variant="primary",
                size="lg",
                elem_classes=["convert-cta"]
            )

        # ==================================================
        # RIGHT COLUMN: Live Simulator, Output & PDF Reader
        # ==================================================
        with gr.Column(scale=5):
            live_page_preview = gr.HTML(value=initial_preview_html)

            with gr.Tabs():
                with gr.Tab("📥 Download & Report"):
                    pdf_output = gr.File(label="Converted PDF Download")
                    report_output = gr.HTML(
                        value="""
                        <div style="padding: 18px; border-radius: 12px; border: 1px dashed #94a3b8; text-align: center; color: #64748b; font-size: 13.5px;">
                            Your conversion report, size comparison, and download card will appear here after conversion.
                        </div>
                        """
                    )

                with gr.Tab("📖 Built-in PDF Reader"):
                    pdf_reader_output = gr.HTML(
                        value="""
                        <div style="padding: 28px; border-radius: 12px; border: 1px dashed #94a3b8; text-align: center; color: #64748b; font-size: 13.5px;">
                            Convert an eBook to preview and scroll through your rendered PDF right inside this tab.
                        </div>
                        """
                    )

                with gr.Tab("🗂️ Session History"):
                    history_table = gr.Dataframe(
                        headers=["Time", "Source File", "Output PDF", "Profile", "Size", "Duration"],
                        datatype=["str", "str", "str", "str", "str", "str"],
                        interactive=False,
                        label="Books Converted in This Session"
                    )

            with gr.Accordion("🛠️ Live Calibre CLI Command Preview", open=False):
                cli_preview = gr.Markdown(value=initial_cli_md)

    # ======================================================
    # Interactive Event Wiring
    # ======================================================
    all_controls = [
        file_input,
        paper_size,
        m_top,
        m_bottom,
        m_left,
        m_right,
        base_font,
        remove_spacing,
        insert_blank,
        blank_line_size,
        justify,
    ]

    # 1. File diagnostics on upload
    file_input.change(
        fn=inspect_uploaded_file,
        inputs=file_input,
        outputs=file_status_html
    )

    # 2. Margin sync & quick margin buttons
    sync_margins.change(
        fn=on_toggle_margin_sync,
        inputs=[sync_margins, m_top],
        outputs=[m_bottom, m_left, m_right]
    )
    m_top.change(
        fn=on_top_margin_change,
        inputs=[sync_margins, m_top],
        outputs=[m_bottom, m_left, m_right]
    )
    btn_m_compact.click(
        fn=lambda: set_quick_margins(36),
        outputs=[sync_margins, m_top, m_bottom, m_left, m_right]
    )
    btn_m_standard.click(
        fn=lambda: set_quick_margins(54),
        outputs=[sync_margins, m_top, m_bottom, m_left, m_right]
    )
    btn_m_wide.click(
        fn=lambda: set_quick_margins(72),
        outputs=[sync_margins, m_top, m_bottom, m_left, m_right]
    )

    # 3. Conditional blank line slider
    insert_blank.change(
        fn=lambda checked: gr.update(interactive=checked),
        inputs=insert_blank,
        outputs=blank_line_size
    )

    # 4. Smart Presets & Reset
    preset_outputs = [
        paper_size, sync_margins, m_top, m_bottom, m_left, m_right,
        base_font, remove_spacing, insert_blank, blank_line_size, justify
    ]
    btn_fiction.click(fn=lambda: apply_preset("fiction"), outputs=preset_outputs)
    btn_academic.click(fn=lambda: apply_preset("academic"), outputs=preset_outputs)
    btn_print.click(fn=lambda: apply_preset("print"), outputs=preset_outputs)
    btn_comfort.click(fn=lambda: apply_preset("comfort"), outputs=preset_outputs)
    btn_reset.click(fn=reset_to_defaults, outputs=preset_outputs)

    # 5. Live Page & CLI Preview updates whenever any setting changes
    for ctrl in all_controls:
        ctrl.change(
            fn=build_live_previews,
            inputs=all_controls,
            outputs=[live_page_preview, cli_preview]
        )

    # 6. Execute Conversion
    convert_btn.click(
        fn=run_conversion_ui,
        inputs=all_controls + [history_state],
        outputs=[pdf_output, report_output, pdf_reader_output, history_table, history_state]
    )

launch_kwargs = {
    "share": True,
    "debug": False,
    "allowed_paths": [str(OUTPUT_DIR.resolve())]
}
if "theme" in inspect.signature(demo.launch).parameters:
    launch_kwargs["theme"] = gr.themes.Soft(primary_hue="indigo", neutral_hue="slate")
    launch_kwargs["css"] = CUSTOM_CSS

demo.launch(**launch_kwargs)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://09d5f85c84e086b154.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
